# Text Classification from Scratch: TF-IDF and Naive Bayes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/nlp/tfidf_naive_bayes.ipynb)

Build a text classifier that sorts raw documents into 20 categories using TF-IDF and Naive Bayes.

**Blog post:** [sesen.ai/blog/text-classification-tfidf-naive-bayes](https://sesen.ai/blog/text-classification-tfidf-naive-bayes)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## 1. Load the Data

In [ ]:
twenty_train = fetch_20newsgroups(subset='train', shuffle=True, random_state=42)
twenty_test = fetch_20newsgroups(subset='test', shuffle=True, random_state=42)

print(f'Training documents: {len(twenty_train.data)}')
print(f'Test documents: {len(twenty_test.data)}')
print(f'Categories: {len(twenty_train.target_names)}')
print()
for i, name in enumerate(twenty_train.target_names):
    print(f'  {i:2d}: {name}')

In [ ]:
print(twenty_train.data[0][:500])
print(f'\nCategory: {twenty_train.target_names[twenty_train.target[0]]}')

## 2. The Quick Win: 10 Lines to 77% Accuracy

In [ ]:
text_clf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', MultinomialNB()),
])

text_clf.fit(twenty_train.data, twenty_train.target)
predicted = text_clf.predict(twenty_test.data)
print(f'Accuracy: {accuracy_score(twenty_test.target, predicted):.1%}')

In [ ]:
docs_new = [
    'OpenGL shading techniques for real-time rendering',
    'The Detroit Tigers signed a new pitcher today',
    'NASA launched the James Webb telescope last year',
    'Is there evidence for the existence of God?',
]

predicted_new = text_clf.predict(docs_new)
for doc, category in zip(docs_new, predicted_new):
    print(f'{twenty_train.target_names[category]:>28s}  \u2190  {doc}')

## 3. Understanding the Pipeline

### Step 1: Bag of Words (CountVectorizer)

In [ ]:
corpus = [
    'The cat sat on the mat',
    'The dog sat on the log',
    'The cat chased the dog',
]
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus)

print('Vocabulary:', vectorizer.get_feature_names_out())
print('\nDocument-term matrix:')
print(X.toarray())

### Step 2: TF-IDF Weighting

In [ ]:
tfidf = TfidfTransformer()
X_tfidf = tfidf.fit_transform(X)

feature_names = vectorizer.get_feature_names_out()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Raw counts
im0 = axes[0].imshow(X.toarray(), cmap='YlOrRd', aspect='auto')
axes[0].set_xticks(range(len(feature_names)))
axes[0].set_xticklabels(feature_names, rotation=45, ha='right', fontsize=11)
axes[0].set_yticks(range(3))
axes[0].set_yticklabels(['Doc 1: cat/mat', 'Doc 2: dog/log', 'Doc 3: cat/dog'], fontsize=10)
axes[0].set_title('Raw Word Counts', fontsize=12)
for i in range(3):
    for j in range(len(feature_names)):
        axes[0].text(j, i, f'{X.toarray()[i,j]:.0f}', ha='center', va='center', fontsize=11,
                     color='white' if X.toarray()[i,j] > 1 else 'black')
plt.colorbar(im0, ax=axes[0], shrink=0.8)

# TF-IDF
im1 = axes[1].imshow(X_tfidf.toarray(), cmap='YlOrRd', aspect='auto')
axes[1].set_xticks(range(len(feature_names)))
axes[1].set_xticklabels(feature_names, rotation=45, ha='right', fontsize=11)
axes[1].set_yticks(range(3))
axes[1].set_yticklabels(['Doc 1: cat/mat', 'Doc 2: dog/log', 'Doc 3: cat/dog'], fontsize=10)
axes[1].set_title('TF-IDF Weights', fontsize=12)
for i in range(3):
    for j in range(len(feature_names)):
        val = X_tfidf.toarray()[i,j]
        axes[1].text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=10,
                     color='white' if val > 0.4 else 'black')
plt.colorbar(im1, ax=axes[1], shrink=0.8)

fig.suptitle('Bag of Words \u2192 TF-IDF: Common words get down-weighted', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### Step 3: Naive Bayes Classification

In [ ]:
cm = confusion_matrix(twenty_test.target, predicted)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

short_names = [name.split('.')[-1] if '.' in name else name for name in twenty_train.target_names]

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(cm_norm, interpolation='nearest', cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(20))
ax.set_yticks(range(20))
ax.set_xticklabels(short_names, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(short_names, fontsize=9)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title('Naive Bayes Confusion Matrix (normalised)', fontsize=14)
plt.colorbar(im, ax=ax, shrink=0.8, label='Fraction')
plt.tight_layout()
plt.show()

## 4. Improving the Baseline

### Stop Words

In [ ]:
text_clf_stop = Pipeline([
    ('vect', CountVectorizer(stop_words='english')),
    ('tfidf', TfidfTransformer()),
    ('clf', MultinomialNB()),
])
text_clf_stop.fit(twenty_train.data, twenty_train.target)
predicted_stop = text_clf_stop.predict(twenty_test.data)
print(f'NB + stop words: {accuracy_score(twenty_test.target, predicted_stop):.1%}')

### SVM Comparison

In [ ]:
text_clf_svm = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf-svm', SGDClassifier(loss='hinge', penalty='l2',
                               alpha=1e-3, max_iter=100,
                               random_state=42)),
])
text_clf_svm.fit(twenty_train.data, twenty_train.target)
predicted_svm = text_clf_svm.predict(twenty_test.data)
print(f'SVM accuracy: {accuracy_score(twenty_test.target, predicted_svm):.1%}')

### Grid Search

In [ ]:
parameters = {
    'vect__ngram_range': [(1, 1), (1, 2)],
    'tfidf__use_idf': (True, False),
    'clf__alpha': (1e-2, 1e-3),
}

gs_clf = GridSearchCV(text_clf, parameters, cv=5, n_jobs=-1)
gs_clf.fit(twenty_train.data, twenty_train.target)

print(f'Best CV score: {gs_clf.best_score_:.1%}')
print(f'Best params: {gs_clf.best_params_}')
print(f'Test accuracy: {accuracy_score(twenty_test.target, gs_clf.predict(twenty_test.data)):.1%}')

In [ ]:
parameters_svm = {
    'vect__ngram_range': [(1, 1), (1, 2)],
    'tfidf__use_idf': (True, False),
    'clf-svm__alpha': (1e-2, 1e-3),
}

gs_svm = GridSearchCV(text_clf_svm, parameters_svm, cv=5, n_jobs=-1)
gs_svm.fit(twenty_train.data, twenty_train.target)

print(f'Best CV score: {gs_svm.best_score_:.1%}')
print(f'Best params: {gs_svm.best_params_}')
print(f'Test accuracy: {accuracy_score(twenty_test.target, gs_svm.predict(twenty_test.data)):.1%}')

In [ ]:
acc_nb = accuracy_score(twenty_test.target, predicted)
acc_gs_nb = accuracy_score(twenty_test.target, gs_clf.predict(twenty_test.data))
acc_gs_svm = accuracy_score(twenty_test.target, gs_svm.predict(twenty_test.data))

methods = ['NB\nbaseline', 'NB +\nstop words', 'SVM\nbaseline', 'NB\ntuned', 'SVM\ntuned']
accuracies = [acc_nb, accuracy_score(twenty_test.target, predicted_stop), acc_svm, acc_gs_nb, acc_gs_svm]
colors = ['#4A90D9', '#5BA0E0', '#E8833A', '#3578C4', '#D06A28']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(methods, [a * 100 for a in accuracies], color=colors, edgecolor='white', linewidth=1.5, width=0.65)
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{acc:.1%}',
            ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_ylim(70, 90)
ax.set_title('Text Classification: Method Comparison on 20 Newsgroups', fontsize=13)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## 5. What the Model Learns

In [ ]:
tfidf_vect = TfidfVectorizer(stop_words='english', max_df=0.9, min_df=5)
X_tfidf_full = tfidf_vect.fit_transform(twenty_train.data)
clf_disc = MultinomialNB().fit(X_tfidf_full, twenty_train.target)

feature_names_full = np.array(tfidf_vect.get_feature_names_out())
log_probs = clf_disc.feature_log_prob_
mean_log_prob = np.mean(log_probs, axis=0)
discriminativeness = log_probs - mean_log_prob

categories_to_show = ['comp.graphics', 'rec.sport.baseball', 'sci.space', 'talk.politics.mideast']
cat_indices = [twenty_train.target_names.index(c) for c in categories_to_show]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()
cat_colors = ['#4A90D9', '#2ECC71', '#E74C3C', '#F39C12']

for ax, cat_idx, cat_name, color in zip(axes, cat_indices, categories_to_show, cat_colors):
    top_indices = discriminativeness[cat_idx].argsort()[-10:]
    top_words = feature_names_full[top_indices]
    top_scores = discriminativeness[cat_idx][top_indices]
    top_scores_norm = (top_scores - top_scores.min()) / (top_scores.max() - top_scores.min())

    ax.barh(range(10), top_scores_norm, color=color, alpha=0.85)
    ax.set_yticks(range(10))
    ax.set_yticklabels(top_words, fontsize=10)
    short_name = cat_name.split('.')[-1] if '.' in cat_name else cat_name
    ax.set_title(short_name, fontsize=12, fontweight='bold')
    ax.set_xlim(0, 1.15)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('Most Discriminative Words per Category', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 6. Stemming

In [ ]:
import nltk
from nltk.stem.snowball import SnowballStemmer

nltk.download('punkt', quiet=True)
stemmer = SnowballStemmer('english', ignore_stopwords=True)

class StemmedCountVectorizer(CountVectorizer):
    def build_analyzer(self):
        analyzer = super().build_analyzer()
        return lambda doc: [stemmer.stem(w) for w in analyzer(doc)]

text_clf_stemmed = Pipeline([
    ('vect', StemmedCountVectorizer(stop_words='english')),
    ('tfidf', TfidfTransformer()),
    ('clf', MultinomialNB(fit_prior=False)),
])
text_clf_stemmed.fit(twenty_train.data, twenty_train.target)
predicted_stemmed = text_clf_stemmed.predict(twenty_test.data)
print(f'NB + stemming + stop words: {accuracy_score(twenty_test.target, predicted_stemmed):.1%}')

## Exercises

1. **Subset classification**: Use only 4 categories (`categories=['comp.graphics', 'rec.sport.baseball', 'sci.space', 'talk.politics.mideast']` in `fetch_20newsgroups`). How much does accuracy improve with fewer, more distinct categories?

2. **Feature engineering**: Add `min_df=5` and `max_df=0.5` to `CountVectorizer` to trim rare and ubiquitous words. How does this affect accuracy and vocabulary size?

3. **Bernoulli vs Multinomial**: Replace `MultinomialNB` with `BernoulliNB`. Does the McCallum & Nigam finding hold on this dataset?

4. **Beyond bag-of-words**: Use `TfidfVectorizer` with `sublinear_tf=True` and character n-grams (`analyzer='char_wb'`, `ngram_range=(3,5)`). Character n-grams capture morphological patterns that word-level features miss.

In [ ]:
# Exercise 1: Subset classification
categories_subset = ['comp.graphics', 'rec.sport.baseball', 'sci.space', 'talk.politics.mideast']
train_sub = fetch_20newsgroups(subset='train', categories=categories_subset, random_state=42)
test_sub = fetch_20newsgroups(subset='test', categories=categories_subset, random_state=42)

# Your code here: build a pipeline and evaluate

# Exercise 2: Feature engineering
# text_clf_trimmed = Pipeline([
#     ('vect', CountVectorizer(min_df=5, max_df=0.5)),
#     ('tfidf', TfidfTransformer()),
#     ('clf', MultinomialNB()),
# ])

# Exercise 3: BernoulliNB
# from sklearn.naive_bayes import BernoulliNB

# Exercise 4: Character n-grams
# from sklearn.feature_extraction.text import TfidfVectorizer
# tfidf_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), sublinear_tf=True)